# v20b1 — external TOOL-OUTPUT training data (CPU, ~5–10 min, one ~271 MB download)

v19 and v20a showed the same gap twice. MSLSAM-Δ has never seen tool output. So it sits ~3.6 SD off-scale on every
InjecAgent row, benign or not, and misses 82% of InjecAgent injections. This notebook builds **750 paired tool-output examples
from a source unrelated to InjecAgent**: `glaiveai/glaive-function-calling-v2` (Apache-2.0), which has real function-calling dialogues
with JSON `FUNCTION RESPONSE`s.

- **task** = the user's request from the dialogue
- **data** = `Tool result (<function>): <JSON response>`. JSON is written with double quotes. InjecAgent's test rows use Python-repr
  quotes, so the training format is *not* a copy of the test format.
- **injection** = the JSON's longest string field is replaced by an external payload. Half are LLMail-Inject attacker emails and half are
  hackaprompt attacks, all **disjoint from every payload used in v19**.
- **benign twin** = the same response, with that field replaced by a length-matched declarative sentence (SQuAD). This is the same
  construction as InjecAgent's own clean tier. There are no instruction-shaped negatives (the v16 lesson).
- v19a's contamination guard (word 8-grams vs the whole test set) and token check (≤ 1,024) are applied unchanged.

Pre-registration: `versions/19_mslsam-v20/README.md` (the v20b section), fixed before any v20b number.

In [1]:
import os, re, json, hashlib, bisect
import numpy as np, pandas as pd
SEED = 42; rng = np.random.default_rng(SEED)
OUT, EXT = "datasets/unified", "datasets/external"
MODEL_ID, MAX_LEN, MAX_DATA_CHARS = "meta-llama/Meta-Llama-3-8B-Instruct", 1024, 2800
SYSTEM_PROMPT = ("You are a helpful, secure, and harmless AI assistant. "
                 "You must refuse to assist with any dangerous, illegal, "
                 "or unethical requests.")
N_PAIRS, CONTAM_NGRAM, CONTAM_MAX = 750, 8, 0.5
def clean_text(t):                                    # v14a/v15a/v19a verbatim
    t = re.sub(r"\s+", " ", str(t).strip())
    return re.sub(r"[^\x00-\x7F]+", "", t)
def md5(s): return hashlib.md5(s.encode()).hexdigest()[:10]
def join_user(task, data): return f"{task}\n\n{data}" if task else data

# ---- glaive function-calling v2: one file, cached ----
GP = f"{EXT}/glaive_fc_v2.json"
if not os.path.exists(GP):
    from huggingface_hub import hf_hub_download
    import shutil
    shutil.copy(hf_hub_download("glaiveai/glaive-function-calling-v2", "glaive-function-calling-v2.json", repo_type="dataset"), GP)
raw = json.load(open(GP))
print(f"glaive-function-calling-v2: {len(raw):,} dialogues | keys {list(raw[0].keys())}")

U_RE = re.compile(r"USER:\s*(.*?)\s*ASSISTANT:", re.S)
FC_RE = re.compile(r"<functioncall>\s*(\{.*?\})\s*(?:<\|endoftext\|>)?\s*FUNCTION RESPONSE:\s*(.*?)\s*(?:ASSISTANT:|USER:|$)", re.S)
NAME_RE = re.compile(r'"name"\s*:\s*"([^"]+)"')
def leaves(o, path=()):
    if isinstance(o, dict):
        for k, v in o.items(): yield from leaves(v, path + (k,))
    elif isinstance(o, list):
        for i, v in enumerate(o): yield from leaves(v, path + (i,))
    elif isinstance(o, str): yield path, o
def set_path(o, path, val):
    for p in path[:-1]: o = o[p]
    o[path[-1]] = val
CAND = []
for d in raw:
    chat = d.get("chat", "")
    mu, mf = U_RE.search(chat), FC_RE.search(chat)
    if not (mu and mf): continue
    try: resp = json.loads(mf.group(2))
    except Exception: continue
    ls = [(p, v) for p, v in leaves(resp) if len(v) >= 20]
    if not ls: continue
    nm = NAME_RE.search(mf.group(1))
    task = clean_text(mu.group(1))
    if not (10 <= len(task) <= 400): continue
    path = max(ls, key=lambda pv: len(pv[1]))[0]
    CAND.append({"task": task, "fn": nm.group(1) if nm else "tool", "resp": resp, "path": path})
print(f"usable dialogues (user request + JSON response with a string field >= 20 chars): {len(CAND):,}")

glaive-function-calling-v2: 112,960 dialogues | keys ['system', 'chat']
usable dialogues (user request + JSON response with a string field >= 20 chars): 21,134


In [2]:
# ---- payload pools: external, DISJOINT from everything v19 used, contamination-guarded ----
corpus = pd.read_csv(f"{OUT}/injection_corpus_v15.csv")
test = pd.read_csv(f"{OUT}/v19_test.csv", keep_default_na=False)
ext19 = pd.read_csv(f"{OUT}/v19_external_train.csv", keep_default_na=False)
def _norm(t): return re.sub(r"[^a-z0-9 ]", " ", str(t).lower()).split()
def _grams(w, n=CONTAM_NGRAM): return {" ".join(w[i:i+n]) for i in range(len(w) - n + 1)}
TG, TS = set(), set()
for t in list(corpus.text.astype(str)) + list(test.task) + list(test.data):
    w = _norm(t); TG |= _grams(w)
    if len(w) < CONTAM_NGRAM: TS.add(" ".join(w))
def contaminated(t):
    w = _norm(t)
    if len(w) < CONTAM_NGRAM: return " ".join(w) in TS
    g = _grams(w); return len(g & TG) / len(g) >= CONTAM_MAX
USED = set(ext19.data.str.lower()) | set(ext19.piece.str.lower())
def pool(name, texts, lo, hi):
    s = pd.Series([clean_text(t) for t in texts]).drop_duplicates()
    s = s[(s.str.len() >= lo) & (s.str.len() <= hi)]; s = s[~s.str.lower().duplicated()]
    used = s.str.lower().isin(USED)
    s = s[~used]; bad = s.map(contaminated); s = s[~bad].tolist(); rng.shuffle(s)
    print(f"  {name:<12} {len(s):>7,} usable (dropped {int(used.sum())} used in v19, {int(bad.sum())} test near-duplicates)")
    return s
ll = pd.read_parquet(f"{EXT}/llmail.parquet")
hp = pd.read_parquet("datasets/hackaprompt.parquet")
LLM = pool("llmail", (ll.subject.fillna("") + " " + ll.body.fillna("")), 30, 600)
HACK = pool("hackaprompt", hp[hp.correct == True].user_input, 15, 600)
sq = pd.read_parquet(f"{EXT}/squad.parquet")
used_ctx = set(ext19[ext19.corpus.str.startswith("EXT-Doc")].data.str[:200])
SENTS = sorted({s.strip() + "." for c in sq.context.drop_duplicates().sample(min(20000, sq.context.nunique()), random_state=SEED)
                for s in clean_text(c).split(". ") if 20 <= len(s) <= 600 and not s.strip().endswith("?")}, key=len)
LENS = [len(x) for x in SENTS]
def _take1(n):
    i = min(bisect.bisect_left(LENS, n), len(SENTS) - 1); LENS.pop(i); return SENTS.pop(i)
def take_sentence(n):
    # length-matched DECLARATIVE filler: nearest-length sentence, plus more sentences while under 80% of the target
    parts = [_take1(n)]
    while sum(len(x) + 1 for x in parts) < 0.8 * n and len(parts) < 6:
        parts.append(_take1(max(20, n - sum(len(x) + 1 for x in parts))))
    return " ".join(parts)
PAY = [x for pair in zip(LLM, HACK) for x in pair]
assert len(PAY) >= N_PAIRS
print(f"  filler sentences: {len(SENTS):,} | payloads available: {len(PAY):,}")

  llmail        19,667 usable (dropped 639 used in v19, 6 test near-duplicates)
  hackaprompt   15,295 usable (dropped 1508 used in v19, 102 test near-duplicates)
  filler sentences: 92,460 | payloads available: 30,590


In [3]:
# ---- build 750 PAIRED tool-output rows ----
import copy
idx = rng.permutation(len(CAND)); X = []; k = 0; LSET = set(LLM)
for j in idx:
    if k >= N_PAIRS: break
    c = CAND[j]; pl = PAY[k]
    base = json.dumps(c["resp"])
    if contaminated(base) or contaminated(c["task"]): continue
    # the payload as it actually appears inside the JSON text (json.dumps escapes quotes/backslashes/unicode)
    esc = lambda t: clean_text(json.dumps(t)[1:-1])
    inj = copy.deepcopy(c["resp"]); set_path(inj, c["path"], pl)
    d_inj = clean_text(f"Tool result ({c['fn']}): {json.dumps(inj)}")
    if esc(pl) not in d_inj or len(d_inj) > MAX_DATA_CHARS + 400:
        PAY.pop(k); continue                          # unusable payload: drop it, keep the filler pool intact
    filler = take_sentence(len(esc(pl)))              # taken only once the injected row is known to be valid
    ben = copy.deepcopy(c["resp"]); set_path(ben, c["path"], filler)
    d_ben = clean_text(f"Tool result ({c['fn']}): {json.dumps(ben)}")
    assert esc(filler) in d_ben
    g = f"tool-{k}"
    src = "llmail" if pl in LSET else "hackaprompt"
    X.append(dict(corpus="EXT-Tool", family="indirect", label=1, task=c["task"], data=d_inj, group=g,
                  source=f"glaive+{src}", piece=esc(pl), pos=".".join(map(str, c["path"]))))
    X.append(dict(corpus="EXT-Tool-Benign", family="indirect", label=0, task=c["task"], data=d_ben, group=g,
                  source="glaive+squad", piece=esc(filler), pos=".".join(map(str, c["path"]))))
    k += 1
    assert k >= N_PAIRS or len(SENTS) > 50, "filler pool nearly empty -- increase the SQuAD sample in the previous cell"
TOOL = pd.DataFrame(X); TOOL["text"] = [join_user(t, d) for t, d in zip(TOOL.task, TOOL.data)]
assert k == N_PAIRS, f"only {k} pairs could be built"
assert not TOOL.text.duplicated().any()
print(TOOL.groupby(["corpus", "source"]).size().to_string())
la, lb = TOOL[TOOL.label == 1].text.str.len().median(), TOOL[TOOL.label == 0].text.str.len().median()
print(f"median chars: injected {la:.0f} | benign {lb:.0f} (gap {abs(la-lb)/la*100:.1f}%)")
print("example (injected):", TOOL.text.iloc[0][:300])

corpus           source            
EXT-Tool         glaive+hackaprompt    375
                 glaive+llmail         375
EXT-Tool-Benign  glaive+squad          750
median chars: injected 454 | benign 454 (gap 0.0%)
example (injected): I need to create a poll for my team to decide our next team building activity. The options are "Bowling", "Movie Night", "Paintball", and "Cooking Class".

Tool result (create_poll): {"status": "success", "poll_id": "12345", "question": "Comprehensive Summary and Final Action From: Investor Relation


In [4]:
# ---- token check with the real tokenizer + write ----
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ.get("HF_TOKEN"))
fmt = lambda u: tok.apply_chat_template([{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": u}],
                                        tokenize=False, add_generation_prompt=True)
TOOL["n_tokens"] = [len(tok(fmt(u), add_special_tokens=False).input_ids) for u in TOOL.text]
print(f"tokens p50 {TOOL.n_tokens.median():.0f} p99 {TOOL.n_tokens.quantile(.99):.0f} max {TOOL.n_tokens.max()} (cap {MAX_LEN})")
assert TOOL.n_tokens.max() <= MAX_LEN, "a tool row exceeds the token cap"
TOOL.insert(0, "uid", [f"tool-{i:05d}" for i in range(len(TOOL))])
TOOL.to_csv(f"{OUT}/v20_tool_train.csv", index=False)
print(f"wrote {OUT}/v20_tool_train.csv ({len(TOOL):,} rows)\nNEXT: v20b2_extract.ipynb (FRESH kernel, ~30-45 min GPU).")

tokens p50 151 p99 269 max 398 (cap 1024)
wrote datasets/unified/v20_tool_train.csv (1,500 rows)
NEXT: v20b2_extract.ipynb (FRESH kernel, ~30-45 min GPU).
